# Bundle Adjustment for Camera Calibration Refinement

This notebook implements an optimization pipeline based on Bundle Adjustment (BA) to refine and improve the calibration of a multi-camera setup.

In computer vision, Bundle Adjustment serves as the final geometric refinement step. Its goal is to optimize the camera parameters (both intrinsics and extrinsics).

The core objective is to minimize the reprojection error, ensuring that the mathematical projections of 3D points onto image planes match the actual 2D ground-truth measurements as closely as possible.

In [1]:
%matplotlib widget
import os
import json
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
import ipywidgets as widgets

## 1. Setup and Multi-Camera Data Loading

In this initial step, we import the necessary libraries and define helper functions to load our data. 
We iterate through all available cameras to load their initial calibration parameters and their 
corresponding annotated 2D-3D ground truth points.

**NOTE ABOUT THE DATA USED:** In this script, we used the calibration data contained in the `camera_data_with_Rvecs_2ndversion` folder, along with the court reference points combined from both `camera_data_with_Rvecs` and `camera_data_with_Rvecs_2ndversion`. More specifically, all reference point data originates from `camera_data_with_Rvecs_2ndversion`, except for cameras 3 and 5. For these two cameras:

- The 3D real coordinates (`real_corners`) are taken from `camera_data_with_Rvecs_2ndversion`.
- The corresponding 2D annotations (`img_corners`) are taken from `camera_data_with_Rvecs`.

We adopted this setup because the calibration data in `camera_data_with_Rvecs_2ndversion` was the most realistic and accurate. The 3D real coordinates in `camera_data_with_Rvecs_2ndversion` made much more sense geometricall, for instance, symmetric points on the court retained symmetric coordinates relative to the origin at the field center, whereas in the other folder they were asymmetric, showing a 0.5 m shift on one side.

Conversely, the 2D annotations in `camera_data_with_Rvecs_2ndversion` for cameras 3 and 5 suffered from a misalignment when plotted over the court images. On the other hand, the 2D annotations from `camera_data_with_Rvecs` coincided perfectly with the actual court corners.

For this reason, we created a custom hybrid dataset stored in the `camera_data_custom_for_BA` folder containing the configuration described above.

In [2]:
# --- Functions to load camera parameters --- 
def load_cam_params(json_path):
    ''' 
    Loads camera parameters from a JSON file and computes matrices and vectors.

    Args:
        json_path (str): Path to the JSON file containing camera parameters.

    Returns:
        P (np.ndarray): Camera matrix.
        K (np.ndarray): Intrinsic camera parameters matrix.
        dist (np.ndarray): Distortion coefficients.
        rvec (np.ndarray): Rotation vector (Rodrigues).
        tvec (np.ndarray): Translation vector.
    '''
    with open(json_path, 'r') as f:
        data = json.load(f)

        # Cast to float32 to ensure compatibility with OpenCV and SciPy
        K = np.array(data['mtx'], dtype=np.float32) 
        dist = np.array(data['dist'], dtype=np.float32) 
        rvec = np.array(data['rvecs'], dtype=np.float32) 
        tvec = np.array(data['tvecs'], dtype=np.float32) 

        R, _ = cv2.Rodrigues(rvec) 
        Rt = np.hstack((R, tvec))  
        P = np.dot(K, Rt) 

        return P, K, dist, rvec, tvec

def load_all_cams_params(cams_id, base_dir):
    '''
    Loads camera parameters for all specified cameras.
    '''
    P_dict, K_dict, dist_dict, rvec_dict, tvec_dict = {}, {}, {}, {}, {}
    
    for cam_id in cams_id:
        json_path = os.path.join(base_dir, f"cam_{cam_id}", "calib", "camera_calib.json")
        #json_path = os.path.join(base_dir, f"cam_{cam_id}", "calib", "camera_calib_real.json")
        
        if not os.path.exists(json_path):
            print(f"[WARNING] Calib file not found for Cam {cam_id}: {json_path}")
            continue
            
        try:
            P, K, dist, rvec, tvec = load_cam_params(json_path)
            P_dict[cam_id] = P
            K_dict[cam_id] = K
            dist_dict[cam_id] = dist
            rvec_dict[cam_id] = rvec
            tvec_dict[cam_id] = tvec
        except Exception as e:
            print(f"[ERROR] Failed to load parameters for Cam {cam_id}: {e}")
            
    return P_dict, K_dict, dist_dict, rvec_dict, tvec_dict


# --- Functions to load 2D-3D field points correspondences ---
def load_all_cams_points(cams_id, base_dir):
    '''
    Loads 2D-3D ground truth point correspondences for all specified cameras.
    Converts 3D coordinates from meters to millimeters to ensure consistency.

    Args:
        cams_id (list): List of camera IDs.
        base_dir (str): Base directory containing camera data.

    Returns:
        pts_3d_dict (dict): Dictionary of 3D point coordinates.
        pts_2d_dict (dict): Dictionary of 2D point coordinates.
    '''
    pts_3d_dict, pts_2d_dict = {}, {}
    
    for cam_id in cams_id:
        json_path = os.path.join(base_dir, f"cam_{cam_id}", "calib", "img_points.json")
        
        if not os.path.exists(json_path):
            print(f"[WARNING] Points file not found for Cam {cam_id}: {json_path}")
            continue
            
        with open(json_path, 'r') as f:
            corners_data = json.load(f)
            
            # Extract and convert scales
            points_3d = np.array(corners_data['real_corners'], dtype=np.float32) * 1000.0 # mm conversion
            points_2d = np.array(corners_data['img_corners'], dtype=np.float32)
            
            pts_3d_dict[cam_id] = points_3d
            pts_2d_dict[cam_id] = points_2d
            
    return pts_3d_dict, pts_2d_dict



# Path Configuration 
SCRIPT_DIR = Path.cwd()
BASE_CALIB_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, "camera_data_custom_for_BA", "camera_data"))
#BASE_CALIB_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, "camera_data_with_Rvecs_2ndversion", "Camera_config2"))
#BASE_CALIB_DIR = os.path.abspath(os.path.join(SCRIPT_DIR, "camera_data_with_Rvecs", "camera_data"))

# Define available cameras in the setup
CAMS_ID = ["1", "2", "3", "4", "5", "7"]


# Create a dictionary to hold all camera parameters and points
print("[INFO] Loading camera parameters...")
P_dict, K_dict, dist_dict, rvec_dict, tvec_dict = load_all_cams_params(CAMS_ID, BASE_CALIB_DIR)

print("[INFO] Loading ground truth points...")
pts_3d_dict, pts_2d_dict = load_all_cams_points(CAMS_ID, BASE_CALIB_DIR)

print(f"\n[SUMMARY] Data successfully loaded for {len(K_dict)} cameras.")

[INFO] Loading camera parameters...
[INFO] Loading ground truth points...

[SUMMARY] Data successfully loaded for 6 cameras.


## 2. Optimization Helper Functions

The core of Bundle Adjustment is the optimization algorithm (Levenberg-Marquardt). However, optimization libraries like `scipy.optimize` do not understand OpenCV's multidimensional matrices; they require a single, flat **1D array** of parameters to modify.

In this step, we define three crucial functions:
1. `pack_parameters`: Flattens the camera parameters (`rvec`, `tvec`, focal lengths, optical center, and distortion coefficients) into a single 15-element list.
2. `unpack_parameters`: Performs the reverse operation. It takes the 1D array generated by the optimizer at each iteration and rebuilds the standard OpenCV matrices.
3. `compute_residuals`: This function defines the cost to minimize durind the BA execution. It projects the 3D points using the current parameter guess and calculates the distance in pixels (the error) compared to 2D ground truth annotations.

Before flattening the camera paraemters are in the shape:

Rotation Vector (`rvec`): A $3 \times 1$ vector using the Rodrigues axis-angle representation. The direction of the vector defines the 3D axis the camera rotates around. The length (norm) of the vector defines the angle of rotation in radians.
$$ \text{rvec} = \begin{bmatrix} r_x & r_y & r_z \end{bmatrix}^T $$

Translation Vector (`tvec`): A $3 \times 1$ spatial vector. It represents the physical 3D position offset of the camera relative to the world origin, measured along the X, Y, and Z axes (scaled in millimeters).

$$ \text{tvec} = \begin{bmatrix} t_x & t_y & t_z \end{bmatrix}^T $$
*  Meaning: 
Intrinsic Matrix (`K`): A $3 \times 3$ matrix defining the internal geometry of the camera.
$$ K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix} $$
*   **$f_x, f_y$**: The focal lengths expressed in pixel units.
*   **$c_x, c_y$**: The principal point (optical center), representing the pixel coordinates where the optical axis intersects the image plane (usually near the center of the image).

Distortion Coefficients (`dist`): A $1 \times 5$ vector correcting lens imperfections.
$$ \text{dist} = \begin{bmatrix} k_1 & k_2 & p_1 & p_2 & k_3 \end{bmatrix} $$
*   $k_1, k_2, k_3$: Radial distortion coefficients. They correct the "barrel" or "pincushion" effect where straight lines curve near the image edges.
*   $p_1, p_2$: Tangential distortion coefficients. They correct the misalignment that occurs when the camera lens is not perfectly parallel to the imaging sensor.


### The Objective Function: `compute_residuals`

The `compute_residuals` function represents the mathematical heart of our Bundle Adjustment. It calculates the reprojection error, which is the physical distance (in pixels) between where the 3D points are projected using the optimizer's current guess, and where they actually are in the ground truth annotations.

Mathematically, let $X_i$ be a 3D world point (`points_3d`) and $x_i$ be its corresponding 2D ground truth pixel location (`points_2d_gt`). The camera projection model predicts the 2D position $\hat{x}_i$ as:
$$ \hat{x}_i = \pi(X_i; K, \text{dist}, \text{rvec}, \text{tvec}) $$

Our function computes the residual vector (the error) for each point:
$$ r_i = \hat{x}_i - x_i $$

The SciPy `least_squares` algorithm takes this array of residuals and applies the Levenberg-Marquardt algorithm to find the optimal camera parameters $\theta^*$ that minimize the total squared sum of these errors. This is the classic Bundle Adjustment cost function:
$$ \boldsymbol{\theta}^* = \arg\min_{\boldsymbol{\theta}} \frac{1}{2} \sum_{i=1}^{N} \| \pi(X_i; \boldsymbol{\theta}) - x_i \|^2 $$


**NOTE:**  $\theta$ could be all the camera parameters or only the extrinsics. This can be choosen by setting the flag `extrinsics_only = True/False` in the created functions.

In [3]:
def pack_parameters(rvec, tvec, K=None, dist=None, extrinsics_only=False):
    '''
    Flattens camera parameters into a 1D array for the SciPy optimizer.
    
    Args:
        rvec (np.ndarray): Rotation vector (3x1).
        tvec (np.ndarray): Translation vector (3x1).
        K (np.ndarray, optional): Intrinsic matrix (3x3).
        dist (np.ndarray, optional): Distortion coefficients (1x5).
        extrinsics_only (bool): If True, packs only rvec and tvec (6 elements).
    
    Returns:
        np.ndarray: A 1D array of length 6 or 15.
    '''
    # If only extrinsics are needed, return just rvec and tvec
    if extrinsics_only:
        return np.hstack((rvec.ravel(), tvec.ravel()))
    

    # Extract intrinsic components from the 3x3 K matrix
    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    intrinsics = np.array([fx, fy, cx, cy], dtype=np.float32)
    
    # Ensure distortion is a flat 1D array of exactly 5 elements
    dist_flat = dist.ravel()[:5]
    
    # Concatenate all components into a single 1D array
    return np.hstack((rvec.ravel(), tvec.ravel(), intrinsics, dist_flat))


def unpack_parameters(params, K_ref=None, dist_ref=None, extrinsics_only=False):
    '''
    Reconstructs OpenCV-compatible matrices from the 1D parameter array.
    
    Args:
        params (np.ndarray): 1D array of parameters.
        K_ref (np.ndarray, optional): Fixed intrinsic matrix (used when extrinsics_only=True).
        dist_ref (np.ndarray, optional): Fixed distortion array (used when extrinsics_only=True).
        extrinsics_only (bool): Flag indicating if params contains 6 or 15 elements.
        
    Returns:
        rvec, tvec, K, dist matrices.
    '''

    # Extract extrinsics (3x1 vectors)
    rvec = params[0:3].reshape(3, 1)
    tvec = params[3:6].reshape(3, 1)

    # If only extrinsics are being optimized, return them alongside the fixed intrinsics
    if extrinsics_only:
        # Return optimized extrinsics alongside original fixed intrinsics
        return rvec, tvec, K_ref, dist_ref
    
    # Reconstruct the 3x3 Intrinsic matrix (K)
    fx, fy, cx, cy = params[6:10]
    K = np.array([
        [fx,  0, cx],
        [ 0, fy, cy],
        [ 0,  0,  1]
    ], dtype=np.float32)
    
    # Reconstruct distortion coefficients (1x5 vector)
    dist = params[10:15].reshape(1, 5)
    
    return rvec, tvec, K, dist


def compute_residuals(params, points_3d, points_2d_gt, K_ref=None, dist_ref=None, extrinsics_only=False):
    '''
    The objective function for the Bundle Adjustment.
    Computes the geometric difference between projected points and ground truth annotations.
    
    Args:
        params (np.ndarray): The current 1D array of camera parameters.
        points_3d (np.ndarray): 3D world points in millimeters.
        points_2d_gt (np.ndarray): 2D ground truth pixel coordinates.
        K_ref (np.ndarray, optional): Fixed intrinsic matrix, used when extrinsics_only=True.
        dist_ref (np.ndarray, optional): Fixed distortion array, used when extrinsics_only=True.
        extrinsics_only (bool): Flag indicating if params contains 6 or 15 elements.

    Returns:
        np.ndarray: A flattened 1D array of projection errors.
    '''
    # 1. Rebuild matrices from the current optimizer guess
    rvec, tvec, K, dist = unpack_parameters(params, K_ref, dist_ref, extrinsics_only)
    
    # 2. Project 3D points to the 2D image plane using current parameters
    # Note: cv2.projectPoints automatically applies the distortion coefficients
    proj_2d, _ = cv2.projectPoints(points_3d, rvec, tvec, K, dist)
    proj_2d = proj_2d.reshape(-1, 2)
    
    # 3. Calculate the error (distance vector) for each point
    error = proj_2d - points_2d_gt
    
    # SciPy's least_squares requires a flat 1D array of residuals
    return error.ravel()

## 3. Multi-Camera Optimization Loop (Bundle Adjustment)

With our parameter transformation and residual functions ready, we now run the Bundle Adjustment algorithm across all available cameras using the Trust Region Reflective solver (`method='trf'`).

For each camera in our setup, the optimization pipeline executes the following sequence:
1. Initial Evaluation: Packs the camera matrices into a 1D vector and calculates the baseline Root Mean Square Error (RMSE) in pixels.
2. Non-Linear Optimization: Executes `scipy.optimize.least_squares` to iteratively tune $rvec$, $tvec$, $K$, and $dist$ until reprojection error converges.
3. Reconstruction and Storage: Unpacks the refined 1D vector back into standard OpenCV matrix structures, re-computes the updated camera matrix $P_{\text{opt}} = K_{\text{opt}} [R_{\text{opt}} \mid t_{\text{opt}}]$, and logs the final error reduction.


### The objective function and geometric reprojection error

In camera calibration, the optimization procedure aims to minimize the geometric reprojection error. This metric quantifies the Euclidean distance in the image plane between an observed two-dimensional coordinate, representing the ground truth, and the corresponding projection of a three-dimensional world point.

Let $\mathbf{x}_i \in \mathbb{R}^2$ denote the observed two-dimensional point and $\mathbf{X}_i \in \mathbb{R}^3$ the corresponding three-dimensional world point. The non-linear projection function is defined as $\pi(\mathbf{X}_i; \mathbf{\theta})$, where $\mathbf{\theta} \in \mathbb{R}^{15}$ represents the camera parameter vector, so $\pi()$ perspectively projects 3D points into the $i$-th camera with distortion. The predicted image coordinate is thus given by:

$$ \hat{\mathbf{x}}_i = \pi(\mathbf{X}_i; \mathbf{\theta}) $$

The residual vector $\mathbf{r}_i(\mathbf{\theta})$ for a single point $i$ is formulated as the difference between the predicted and observed coordinates:

$$ \mathbf{r}_i(\mathbf{\theta}) = \hat{\mathbf{x}}_i - \mathbf{x}_i = \begin{bmatrix} \hat{u}_i - u_i \\ \hat{v}_i - v_i \end{bmatrix} $$

By aggregating all $N$ point residuals into a global residual vector $\mathbf{r}(\mathbf{\theta}) \in \mathbb{R}^{2N}$, the objective function $E(\mathbf{\theta})$ is defined as half the sum of squared residuals:

$$ E(\mathbf{\theta}) = \frac{1}{2} \|\mathbf{r}(\mathbf{\theta})\|^2_2 = \frac{1}{2} \sum_{i=1}^{N} \left( (\hat{u}_i - u_i)^2 + (\hat{v}_i - v_i)^2 \right) $$

The Root Mean Square Error metric evaluated in the script is directly derived from this objective function, representing the square root of the mean squared scalar residual across all $2N$ components:

$$ \text{RMSE} = \sqrt{\frac{1}{2N} \sum_{i=1}^{N} \left( (\hat{u}_i - u_i)^2 + (\hat{v}_i - v_i)^2 \right)} = \sqrt{\frac{E(\mathbf{\theta})}{N}} $$

**Note:** As previously stated, $\mathbf{\theta}$ can include all camera parameters or exclusively the extrinsics. This behavior is controlled via the `extrinsics_only` boolean flag.


### Non-Linear Optimization and Bounded Least Squares

While the Levenberg-Marquardt algorithm is widely used in standard calibration literature, the Trust Region Reflective (TRF) method was selected specifically for its native support of parameter bounds ($\mathbf{\theta}_{lb} \le \mathbf{\theta} \le \mathbf{\theta}_{ub}$). The optimization is formulated as a bounded non-linear least-squares problem:

$$ \min_{\mathbf{\theta}_{lb} \le \mathbf{\theta} \le \mathbf{\theta}_{ub}} \frac{1}{2} \|\mathbf{r}(\mathbf{\theta})\|^2_2 $$

TRF optimizes the objective function within a local trust region, dynamically reflecting the search trajectory whenever a parameter encounters a boundary constraint.

Imposing explicit bounds around the initial parameters $\mathbf{\theta}_0$ prevents the optimizer from overfitting. Without bounds, the optimization may yield parameters that perfectly fit the court calibration points but perform poorly when reconstructing 3D human skeletons.

In [4]:
def create_bounds(initial_params, extrinsics_only=False):
    '''
    Creates bounds for the optimization parameters.
    Bounds are bounded near the initial camera parameters to prevent overfitting.
    
    Args:
        initial_params (np.ndarray): The initial parameter array.
        extrinsics_only (bool): Flag indicating if params contains 6 or 15 elements.

    Returns:
        tuple: Lower and upper bounds for the optimizer.
    '''

    RVEC_TOLL = 0.5  # Allowable rotation variation in radians, approx. 2 degrees
    TVEC_TOLL = 200          # Allowable translation variation in millimeter

    if extrinsics_only:
        # For extrinsics only, allow a reasonable range around the initial values
        lower_bounds = initial_params - np.array([RVEC_TOLL, RVEC_TOLL, RVEC_TOLL, TVEC_TOLL, TVEC_TOLL, TVEC_TOLL])
        upper_bounds = initial_params + np.array([RVEC_TOLL, RVEC_TOLL, RVEC_TOLL, TVEC_TOLL, TVEC_TOLL, TVEC_TOLL])

        return (lower_bounds, upper_bounds)

    # If optimizing all 15 parameters, unpack them to set specific thresholds
    rvec = initial_params[0:3]
    tvec = initial_params[3:6]
    fx, fy, cx, cy = initial_params[6:10]
    dist = initial_params[10:15]

    # --- 1. Extrinsics Bounds ---
    rvec_lb, rvec_ub = rvec - RVEC_TOLL, rvec + RVEC_TOLL
    tvec_lb, tvec_ub = tvec - TVEC_TOLL, tvec + TVEC_TOLL

    # --- 2. Intrinsics Bounds ---
    # Focal Length: Allowable variation of ±20% around the initial values
    FOCAL_MARGIN = 0.015
    fx_lb, fx_ub = fx * (1 - FOCAL_MARGIN), fx * (1 + FOCAL_MARGIN)
    fy_lb, fy_ub = fy * (1 - FOCAL_MARGIN), fy * (1 + FOCAL_MARGIN)

    # Optical Center : confine within a ±50 pixel radius
    PX_MARGIN = 10.0
    cx_lb, cx_ub = cx - PX_MARGIN, cx + PX_MARGIN
    cy_lb, cy_ub = cy - PX_MARGIN, cy + PX_MARGIN

    # --- 3. Distortion Coefficients Bounds [k1, k2, p1, p2, k3] ---
    k1, k2, p1, p2, k3 = dist

    DIST_MARGIN = 0.02  # 10% relative margin for distortion coefficients
    
    # Radial distortion (k1, k2, k3).
    # Use 25% relative margin
    k1_margin = abs(k1) * DIST_MARGIN 
    k2_margin = abs(k2) * DIST_MARGIN
    k3_margin = abs(k3) * DIST_MARGIN
    
    # Tangential distortion (p1, p2): strictly close to zero.
    # Use 25% relative margin1
    p1_margin = abs(p1) * DIST_MARGIN
    p2_margin = abs(p2) * DIST_MARGIN
    
    # --- 4. Reconstruct the bounds arrays ---
    dist_lb = np.array([k1 - k1_margin, k2 - k2_margin, p1 - p1_margin, p2 - p2_margin, k3 - k3_margin])
    dist_ub = np.array([k1 + k1_margin, k2 + k2_margin, p1 + p1_margin, p2 + p2_margin, k3 + k3_margin])

    # Concatenate all boundaries into two flat 1D arrays required by SciPy
    lower = np.hstack((rvec_lb, tvec_lb, [fx_lb, fy_lb, cx_lb, cy_lb], dist_lb))
    upper = np.hstack((rvec_ub, tvec_ub, [fx_ub, fy_ub, cx_ub, cy_ub], dist_ub))

    return (lower, upper)



# ==========================================
# Optimization Loop for Each Camera
# ==========================================

# Set this flag to True to optimize ONLY extrinsics , or False for ALL parameters (15 DoF)
OPTIMIZE_EXTRINSICS_ONLY = False 

opt_K_dict, opt_dist_dict = {}, {}
opt_rvec_dict, opt_tvec_dict = {}, {}
opt_P_dict = {}

mode_name = "Extrinsic-Only (6 Parameters)" if OPTIMIZE_EXTRINSICS_ONLY else "Full (15 Parameters)"
print(f"[INFO] Starting Bundle Adjustment Optimization [{mode_name}]...\n")
print("=" * 70)

for cam_id in CAMS_ID:
    if cam_id not in K_dict or cam_id not in pts_3d_dict:
        print(f"[WARNING] Missing data for Camera {cam_id}. Skipping optimization.")
        continue
        
    print(f"--- Processing Camera {cam_id} ---")
    
    # Extract initial calibration data
    K = K_dict[cam_id]
    dist = dist_dict[cam_id]
    rvec = rvec_dict[cam_id]
    tvec = tvec_dict[cam_id]
    
    points_3d = pts_3d_dict[cam_id]
    points_2d = pts_2d_dict[cam_id]
    
    # Pack parameters according to selected mode
    initial_params = pack_parameters(
        rvec, tvec, K, dist, 
        extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY
    )

    # Ccompute parameter bounds to enforce physical constraints on the solver
    param_bounds = create_bounds(initial_params, extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY)
    
    # Calculate baseline error
    initial_residuals = compute_residuals(
        initial_params, points_3d, points_2d, 
        K_ref=K, dist_ref=dist, 
        extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY
    )
    initial_rmse = np.sqrt(np.mean(initial_residuals**2))
    print(f"    Initial Reprojection Error (RMSE): {initial_rmse:.3f} pixels")
    
    # # Execute Levenberg-Marquardt solver
    # result = least_squares(
    #     compute_residuals,  # Function to minimize (Objective function)
    #     initial_params,     # Initial guess for parameters
    #     method='lm',        # Levenberg-Marquardt algorithm
    #     args=(points_3d, points_2d, K, dist, OPTIMIZE_EXTRINSICS_ONLY), # Arguments for the Objective function
    #     ftol=1e-6,  # Function tolerance
    #     xtol=1e-6,  # Parameter tolerance
    #     diff_step=5e-5  # Finite difference step size for numerical Jacobian approximation
    # )

    # Execute Trust Region Reflective solver with bounds
    result = least_squares(
        compute_residuals,   # Function to minimize (Objective function)
        initial_params,      # Initial guess for parameters
        method='trf',        # Trust Region Reflective algorithm (supports bounding constraints)
        bounds=param_bounds, # Physical parameter bounds to prevent overfitting
        args=(points_3d, points_2d, K, dist, OPTIMIZE_EXTRINSICS_ONLY),
        ftol=1e-6,           # Function tolerance
        xtol=1e-6,           # Parameter tolerance
        diff_step=5e-5       # Finite difference step size
    )
    
    # Unpack optimized parameters
    optimized_params = result.x
    final_residuals = compute_residuals(
        optimized_params, points_3d, points_2d, 
        K_ref=K, dist_ref=dist, 
        extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY
    )
    final_rmse = np.sqrt(np.mean(final_residuals**2))
    
    rvec_opt, tvec_opt, K_opt, dist_opt = unpack_parameters(
        optimized_params, 
        K_ref=K, dist_ref=dist, 
        extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY
    )
    
    # Reconstruct updated Projection Matrix P
    R_opt, _ = cv2.Rodrigues(rvec_opt)
    Rt_opt = np.hstack((R_opt, tvec_opt))
    P_opt = np.dot(K_opt, Rt_opt)
    
    # Store results
    opt_K_dict[cam_id] = K_opt
    opt_dist_dict[cam_id] = dist_opt
    opt_rvec_dict[cam_id] = rvec_opt
    opt_tvec_dict[cam_id] = tvec_opt
    opt_P_dict[cam_id] = P_opt
    
    print(f"    Final Reprojection Error (RMSE):   {final_rmse:.3f} pixels")
    print(f"    Error Reduction:                  {initial_rmse - final_rmse:.3f} pixels")
    print(f"    Optimization Status:              {result.message}")
    print("=" * 70)

print("\n[INFO] Optimization process finished.")

[INFO] Starting Bundle Adjustment Optimization [Full (15 Parameters)]...

--- Processing Camera 1 ---
    Initial Reprojection Error (RMSE): 2.769 pixels
    Final Reprojection Error (RMSE):   2.557 pixels
    Error Reduction:                  0.213 pixels
    Optimization Status:              `xtol` termination condition is satisfied.
--- Processing Camera 2 ---
    Initial Reprojection Error (RMSE): 6.971 pixels
    Final Reprojection Error (RMSE):   6.245 pixels
    Error Reduction:                  0.726 pixels
    Optimization Status:              `xtol` termination condition is satisfied.
--- Processing Camera 3 ---
    Initial Reprojection Error (RMSE): 24.454 pixels
    Final Reprojection Error (RMSE):   6.027 pixels
    Error Reduction:                  18.427 pixels
    Optimization Status:              `xtol` termination condition is satisfied.
--- Processing Camera 4 ---
    Initial Reprojection Error (RMSE): 6.359 pixels
    Final Reprojection Error (RMSE):   5.943 pixels


## 6. Validation and Comparison Visualizer

This section evaluates the impact of Bundle Adjustment by comparing Ground Truth annotations, Pre-BA projections, and Post-BA projections for each individual camera view.
The optimization results can be visualized in the interactive widget below.

As in the optimization phase, the quantitative error is measured using the Root Mean Square Error (RMSE), calculated across all 2D residual components of the court reference points for the selected camera:

$$ \text{RMSE} = \sqrt{\frac{1}{2N} \sum_{i=1}^{N} \left( (\hat{u}_i - u_i)^2 + (\hat{v}_i - v_i)^2 \right)} $$

**Note**: All visual overlays and evaluations in this section are performed directly on the undistorted images.

In [ ]:
# ==========================================
# 1. Image Paths Configuration
# ==========================================
CAM_IMAGE_PATHS = {
    "1": 'dataset_roboflow/hpe_01.v1i.coco/out1_frame_0001_png.rf.8bb46e3f3827694aebcba9cbe399a9d0.jpg',
    "2": 'dataset_roboflow/hpe_01.v1i.coco/out2_frame_0001_png.rf.4140ef71af4bcba8d84e4ad82a8b8733.jpg',
    "3": 'dataset_roboflow/hpe_01.v1i.coco/out3_frame_0001_png.rf.0bab75e5e390766f75af8a70f768a56d.jpg',
    "4": 'dataset_roboflow/hpe_01.v1i.coco/out4_frame_0001_png.rf.45a16dcfc4beb1a4311c2e330699847b.jpg',
    "5": 'dataset_roboflow/hpe_01.v1i.coco/out5_frame_0001_png.rf.cab33368b30dc0e78b884969b620c156.jpg',
    "7": 'dataset_roboflow/hpe_01.v1i.coco/out7_frame_0001_png.rf.51899b04ef823fc16aafbe308ea637a5.jpg',
}

# ==========================================
# 2. Visualization Logic
# ==========================================
def render_comparison(cam_id):
    if cam_id not in opt_K_dict:
        print(f"[ERROR] No optimization data found for Camera {cam_id}.")
        return

    pts_3d = pts_3d_dict[cam_id]
    pts_2d_gt = pts_2d_dict[cam_id]
    image_path = CAM_IMAGE_PATHS.get(cam_id, "")
    
    # --- Compute Pre-BA Metrics ---
    params_init = pack_parameters(rvec_dict[cam_id], tvec_dict[cam_id], K_dict[cam_id], dist_dict[cam_id], extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY)
    res_init = compute_residuals(params_init, pts_3d, pts_2d_gt, K_ref=K_dict[cam_id], dist_ref=dist_dict[cam_id], extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY)
    rmse_init = np.sqrt(np.mean(res_init**2))
    
    proj_init, _ = cv2.projectPoints(pts_3d, rvec_dict[cam_id], tvec_dict[cam_id], K_dict[cam_id], dist_dict[cam_id])
    proj_init = proj_init.reshape(-1, 2)
    
    # --- Compute Post-BA Metrics ---
    params_opt = pack_parameters(opt_rvec_dict[cam_id], opt_tvec_dict[cam_id], opt_K_dict[cam_id], opt_dist_dict[cam_id], extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY)
    res_opt = compute_residuals(params_opt, pts_3d, pts_2d_gt, K_ref=opt_K_dict[cam_id], dist_ref=opt_dist_dict[cam_id], extrinsics_only=OPTIMIZE_EXTRINSICS_ONLY)
    rmse_opt = np.sqrt(np.mean(res_opt**2))
    
    proj_opt, _ = cv2.projectPoints(pts_3d, opt_rvec_dict[cam_id], opt_tvec_dict[cam_id], opt_K_dict[cam_id], opt_dist_dict[cam_id])
    proj_opt = proj_opt.reshape(-1, 2)
    
    delta_rmse = rmse_init - rmse_opt

    # --- Image and Plot Setup ---
    if not os.path.exists(image_path):
        print(f"[WARNING] Image not found at path: '{image_path}'.")
        return


    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    
    # Close previous figures to prevent memory leaks and keep the notebook lightweight
    plt.close('all')
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(img)
    
    # --- Point Rendering ---
    gt = np.array(pts_2d_gt)
    
    # Plot all points in a vectorized manner for cleaner and more efficient rendering
    ax.plot(gt[:, 0], gt[:, 1], 'rx', markersize=10, markeredgewidth=2, label='Ground Truth (2D)')
    ax.plot(proj_init[:, 0], proj_init[:, 1], 'bo', markersize=6, markeredgecolor='white', alpha=0.7, label='Pre-BA')
    ax.plot(proj_opt[:, 0], proj_opt[:, 1], 'go', markersize=6, markeredgecolor='white', label='Post-BA')
    
    # Draw dashed lines representing the residual error and add text labels for individual points
    for i in range(len(gt)):
        ax.plot([gt[i, 0], proj_opt[i, 0]], [gt[i, 1], proj_opt[i, 1]], 'y--', alpha=0.5, linewidth=1.0)
        ax.text(proj_opt[i, 0] + 10, proj_opt[i, 1] - 10, f"P{i+1}", color='cyan', fontsize=8, 
                bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
        
    # --- Layout Configuration ---
    title = f"Cam {cam_id} | Pre-BA RMSE: {rmse_init:.3f} px | Post-BA RMSE: {rmse_opt:.3f} px | Improvement: {delta_rmse:+.3f} px"
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9)
    ax.axis('off')
    
    fig.tight_layout()
    plt.show()

# ==========================================
# 3. Interactive Widget Deployment
# ==========================================
valid_cams = [cam for cam in CAMS_ID if cam in opt_K_dict]

print("[INFO] Use the slider to switch cameras and the toolbar below for zooming/panning.")
slider = widgets.SelectionSlider(
    options=valid_cams,
    value=valid_cams[0],
    description='Camera ID:',
    disabled=False,
    continuous_update=False, # Prevents recalculation while dragging; updates only upon release
    orientation='horizontal'
)

widgets.interact(render_comparison, cam_id=slider);

[INFO] Use the slider to switch cameras and the toolbar below for zooming/panning.


interactive(children=(SelectionSlider(continuous_update=False, description='Camera ID:', options=('1', '2', '3…

## 7. Export Optimized Parameters

In this section, we save the refined camera parameters (Intrinsics, Extrinsics, and Distortion coefficients). 

To maintain compatibility with the rest of the pipeline and the loading functions defined earlier, the parameters are saved in a new directory named `camera_data_after_BA`. The internal folder structure strictly mirrors the original one: `camera_data_after_BA/camera_data/cam_<id>/calib/camera_calib.json`.

All numpy arrays are converted to nested Python lists to be correctly serialized into JSON format.

In [6]:
# Define the root output directory
OUTPUT_ROOT_DIR = os.path.join(SCRIPT_DIR, "camera_data_after_BA", "camera_data")

print(f"[INFO] Saving optimized camera parameters to: {OUTPUT_ROOT_DIR}\n")

# Iterate over all cameras that were successfully optimized
saved_count = 0
for cam_id in CAMS_ID:
    # 1. Define the specific directory path for the current camera
    cam_calib_dir = os.path.join(OUTPUT_ROOT_DIR, f"cam_{cam_id}", "calib")
    
    # 2. Create the directories if they don't exist
    os.makedirs(cam_calib_dir, exist_ok=True)
    
    # 3. Format the data to match the original JSON structure
    calib_data = {
        "mtx": opt_K_dict[cam_id].tolist(),
        "dist": opt_dist_dict[cam_id].tolist(),
        "rvecs": opt_rvec_dict[cam_id].tolist(),
        "tvecs": opt_tvec_dict[cam_id].tolist()
    }
    
    # 4. Save to JSON file
    json_path = os.path.join(cam_calib_dir, "camera_calib.json")
    try:
        with open(json_path, 'w') as f:
            json.dump(calib_data, f, indent=4)
        print(f"  [SUCCESS] Saved Cam {cam_id} -> {json_path}")
        saved_count += 1
    except Exception as e:
        print(f"  [ERROR] Failed to save Cam {cam_id}: {e}")

print(f"\n[SUMMARY] Successfully exported {saved_count} camera calibration files.")

[INFO] Saving optimized camera parameters to: /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data

  [SUCCESS] Saved Cam 1 -> /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data/cam_1/calib/camera_calib.json
  [SUCCESS] Saved Cam 2 -> /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data/cam_2/calib/camera_calib.json
  [SUCCESS] Saved Cam 3 -> /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data/cam_3/calib/camera_calib.json
  [SUCCESS] Saved Cam 4 -> /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data/cam_4/calib/camera_calib.json
  [SUCCESS] Saved Cam 5 -> /home/ale_rossi/conci/cv25-26-human-pose-estimation-alessandrorossi-eliasantuari/camera_data_after_BA/camera_data/cam_5/calib/ca